# Experiment 4.1 — Causal positional information for RSNN

Question: does explicit causal elapsed-time / absolute-slot information close the gap between Exp4.0 RSNN and Fixed250+Linear?

Only two Exp4.0 anchor configurations are used: `H=64, tau=1000 ms` and `H=128, tau=250 ms`. The no-position controls are reused from Exp4.0; only scalar and one-hot position cases are newly trained.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'snn').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')

repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from scripts import experiment_4_0_fixed250_temporal_snn as exp40
from scripts import experiment_4_1_causal_position_rsnn as exp41

root = exp41.results_dir(repo_root)
runs_path = root / 'runs.csv'
summary_path = root / 'summary.csv'
paired_path = root / 'paired_position_effects.csv'
eval_dir = root / 'evaluations'
eval_count = len(list(eval_dir.glob('*.json'))) if eval_dir.exists() else 0
ready = runs_path.exists() and summary_path.exists() and paired_path.exists()
print('Artifact root:', root)
print(f'New position-aware evaluations: {eval_count}/{exp41.EXPECTED_NEW_RUNS}')
print('Finalized:', ready)
if ready:
    runs = pd.read_csv(runs_path)
    summary = pd.read_csv(summary_path)
    paired = pd.read_csv(paired_path)
    baseline = json.loads((exp40.results_dir(repo_root) / 'baseline' / 'fixed250_linear.json').read_text())
    display(summary)
    display(paired)
    print('Linear reference test BA:', baseline['metrics']['test']['balanced_accuracy'])
else:
    runs = summary = paired = baseline = None
    print('Run: bash scripts/bash_script/SNN_Bash/submit_exp_4_1_cpu.bash')

In [ ]:
if not ready:
    print('Skipped: position-effect plot requires finalized artifacts.')
else:
    test = runs[runs['split'] == 'test'].copy()
    view = (test.groupby(['hidden_width', 'tau_mem_ms', 'position_encoding'], as_index=False)
                ['valid_count_balanced_accuracy'].mean())
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for (width, tau), frame in view.groupby(['hidden_width', 'tau_mem_ms']):
        order = ['none', 'scalar', 'onehot']
        frame = frame.set_index('position_encoding').reindex(order).reset_index()
        ax.plot(frame['position_encoding'], frame['valid_count_balanced_accuracy'], marker='o', label=f'H={width}, tau={int(tau)} ms')
    ax.axhline(baseline['metrics']['test']['balanced_accuracy'], linestyle='--', label='Fixed250 + Linear')
    ax.set_ylabel('Mean test balanced accuracy')
    ax.set_xlabel('Causal position encoding')
    ax.set_title('Does causal absolute position close the RSNN gap?')
    ax.legend()
    ax.grid(True, alpha=0.25)
    plt.show()

In [ ]:
if not ready:
    print('Skipped: paired deltas require finalized artifacts.')
else:
    delta_cols = ['scalar_minus_none', 'onehot_minus_none']
    display(paired.groupby(['hidden_width', 'tau_mem_ms'])[delta_cols].agg(['mean', 'std']))
    tail = (runs[runs['split'] == 'test']
            .groupby(['hidden_width', 'tau_mem_ms', 'position_encoding'], as_index=False)
            ['tail_spike_fraction'].mean())
    display(tail)